<a href="https://colab.research.google.com/github/emanfatimaa05/code-switching-codesaviours-si26-eman/blob/main/SI26_Week7_Eman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Check GPU Availability

This cell checks whether a GPU is available in Google Colab for faster model training.

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU: Tesla T4


## Connect Google Drive

This cell mounts Google Drive so the dataset and trained model can be accessed and saved permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Locate the Dataset

This cell searches Google Drive to locate the saved code-switching dataset.

In [ ]:
import os

for root, dirs, files in os.walk('/content/drive'):
    for file in files:
        if file == 'dataset.csv':
            print("FOUND:", os.path.join(root, file))

FOUND: /content/drive/MyDrive/dataset.csv


## Load the Code-Switching Dataset

This cell loads the dataset from Google Drive and displays its shape, columns, and first few rows.

In [ ]:
import pandas as pd

# Load dataset from Google Drive
df = pd.read_csv('/content/drive/MyDrive/dataset.csv')

print("Dataset loaded successfully! ✅")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully! ✅
Shape: (1593, 3)

Columns:
['sentence', 'word', 'label']

First 5 rows:


,sentence,word,label
0,Tujhe kisiko kuch prove karne ki zarurat nahi hai,Tujhe,URD
1,Tujhe kisiko kuch prove karne ki zarurat nahi hai,kisiko,URD
2,Tujhe kisiko kuch prove karne ki zarurat nahi hai,kuch,URD
3,Tujhe kisiko kuch prove karne ki zarurat nahi hai,prove,ENG
4,Tujhe kisiko kuch prove karne ki zarurat nahi hai,karne,URD


## Check Label Distribution

This cell checks the distribution of Urdu and English labels and confirms the unique labels present in the dataset.

In [ ]:
print("Label distribution:")
print(df['label'].value_counts())

print("\nUnique labels:")
print(df['label'].unique())

Label distribution:
label
URD    1035
ENG     558
Name: count, dtype: int64

Unique labels:
['URD' 'ENG']


## Inspect Dataset Statistics

This cell checks the total number of word entries, unique sentences, and example sentences with their corresponding labels.

In [ ]:
print("Total word rows:", len(df))
print("Unique sentences:", df['sentence'].nunique())

print("\nExample sentences with labels:")
print(df.groupby('sentence')['label'].apply(list).head(5))

Total word rows: 1593
Unique sentences: 157

Example sentences with labels:
sentence
Aaj class mein teacher ne new topic explain kiya                          [URD, ENG, URD, URD, URD, URD, URD, URD, URD]
Aaj itna kaam tha ke I didn't even get time for lunch                 [URD, URD, URD, URD, URD, ENG, ENG, URD, ENG, ...
Aaj ka din bohot busy tha, had 3 meetings back to back                [URD, URD, URD, URD, ENG, URD, ENG, URD, ENG, ...
Aaj ka sara kaam complete ho gaya, finally I can relax                [URD, URD, URD, URD, ENG, URD, URD, URD, ENG, ...
Aaj main treasury benches par hoon, ab Solution meri zimmedari hai    [URD, URD, ENG, ENG, URD, URD, URD, ENG, URD, ...
Name: label, dtype: object


## Split the Dataset into Training and Testing Sets

This cell groups the words and their labels by sentence and splits the 157 sentences into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Label mapping
label2id = {'URD': 0, 'ENG': 1}
id2label = {0: 'URD', 1: 'ENG'}

# Group words by sentence
sentences = df.groupby('sentence').apply(
    lambda x: {
        'words': x['word'].tolist(),
        'labels': x['label'].tolist()
    }
).tolist()

# Split into training and testing data
train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print(f"Total sentences: {len(sentences)}")
print(f"Training sentences: {len(train_data)}")
print(f"Testing sentences: {len(test_data)}")

Total sentences: 157
Training sentences: 125
Testing sentences: 32


/tmp/ipykernel_329/1577716791.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby('sentence').apply(


## Install Required Libraries

This cell installs the libraries required for model training, dataset handling, tokenization, and evaluation.

In [ ]:
!pip install -q transformers datasets seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Load the Pretrained XLM-RoBERTa Model

This cell loads the pretrained XLM-RoBERTa tokenizer and model and adapts it for two-class token classification: Urdu (URD) and English (ENG).

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "xlm-roberta-base"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained model and adapt it for our 2 labels
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print("XLM-RoBERTa loaded successfully! ✅")
print("Labels:", model.config.id2label)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLM-RoBERTa loaded successfully! ✅
Labels: {0: 'URD', 1: 'ENG'}


## Tokenize Words and Align Labels

This cell tokenizes the words and aligns the original Urdu and English labels with the corresponding tokens produced by the tokenizer.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['words'],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)

        label_ids = []
        previous_word = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)

            previous_word = word_id

        labels.append(label_ids)

    tokenized['labels'] = labels

    return tokenized

print("Tokenization function ready! ✅")

Tokenization function ready! ✅


## Convert Data into Hugging Face Dataset Format

This cell converts the training and testing data into Hugging Face Dataset objects so they can be used by the Trainer.

In [ ]:
from datasets import Dataset

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data]
    })

# Convert training and testing data
train_ds = to_hf_dataset(train_data)
test_ds = to_hf_dataset(test_data)

print("Training dataset:", train_ds)
print("Testing dataset:", test_ds)

Training dataset: Dataset({
    features: ['words', 'labels'],
    num_rows: 125
})
Testing dataset: Dataset({
    features: ['words', 'labels'],
    num_rows: 32
})


## Tokenize the Training and Testing Data

This cell applies the tokenization and label-alignment function to both the training and testing datasets.

In [ ]:
train_ds = train_ds.map(
    tokenize_and_align_labels,
    batched=True
)

test_ds = test_ds.map(
    tokenize_and_align_labels,
    batched=True
)

print("Training dataset tokenized! ✅")
print("Testing dataset tokenized! ✅")

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Training dataset tokenized! ✅
Testing dataset tokenized! ✅


## Configure Model Training

This cell defines the training settings, including the number of epochs, batch sizes, evaluation strategy, and model-saving strategy.

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer configured successfully! ✅")

Trainer configured successfully! ✅


## Train the Language Identification Model

This cell trains the XLM-RoBERTa model for five epochs to classify words as either Urdu or English.

In [ ]:
print("Starting training... 🚀")

trainer.train()

print("Training complete! ✅")

Starting training... 🚀


Epoch,Training Loss,Validation Loss
1,0.548579,0.254570
2,0.236519,0.251918
3,0.224524,0.238752
4,0.193909,0.222869
5,0.168139,0.235968


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete! ✅


## Save the Trained Model

This cell saves the trained model and tokenizer to Google Drive so they can be reused without retraining.

In [ ]:
import os

# Permanent location in Google Drive
MODEL_DIR = "/content/drive/MyDrive/CodeSwitching_Week7_Model"

# Save trained model
trainer.save_model(MODEL_DIR)

# Save tokenizer
tokenizer.save_pretrained(MODEL_DIR)

print("MODEL SAVED TO GOOGLE DRIVE! ✅")
print("Location:", MODEL_DIR)

print("\nSaved files:")
for file in sorted(os.listdir(MODEL_DIR)):
    print("✓", file)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL SAVED TO GOOGLE DRIVE! ✅
Location: /content/drive/MyDrive/CodeSwitching_Week7_Model

Saved files:
✓ config.json
✓ model.safetensors
✓ tokenizer.json
✓ tokenizer_config.json
✓ training_args.bin


## Verify the Saved Model

This cell reloads the saved model and tokenizer from Google Drive to confirm that the trained model was saved correctly.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load the model directly from Google Drive
test_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

test_model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

print("MODEL RELOADED SUCCESSFULLY! ✅")
print("Labels:", test_model.config.id2label)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODEL RELOADED SUCCESSFULLY! ✅
Labels: {0: 'URD', 1: 'ENG'}


## Generate Predictions on the Test Dataset

This cell uses the trained model to generate predictions for the words in the testing dataset. The predicted labels are then used for the final evaluation.

In [ ]:
from sklearn.metrics import classification_report, f1_score

# Flatten the true and predicted labels
true_flat = []
pred_flat = []

for i in range(len(test_ds)):
    for j, true_id in enumerate(test_ds[i]["labels"]):

        # Ignore special tokens
        if true_id != -100:
            true_flat.append(id2label[true_id])
            pred_flat.append(id2label[predicted_labels[i][j]])

# Generate correct URD / ENG report
print(classification_report(
    true_flat,
    pred_flat,
    labels=["URD", "ENG"],
    digits=4
))

# Individual F1 scores
urd_f1 = f1_score(
    true_flat,
    pred_flat,
    labels=["URD"],
    average="macro"
)

eng_f1 = f1_score(
    true_flat,
    pred_flat,
    labels=["ENG"],
    average="macro"
)

overall_f1 = f1_score(
    true_flat,
    pred_flat,
    labels=["URD", "ENG"],
    average="weighted"
)

print(f"URD F1: {urd_f1:.4f}")
print(f"ENG F1: {eng_f1:.4f}")
print(f"Overall F1: {overall_f1:.4f}")

              precision    recall  f1-score   support

         URD     0.9896    0.8756    0.9291       217
         ENG     0.7874    0.9804    0.8734       102

    accuracy                         0.9091       319
   macro avg     0.8885    0.9280    0.9012       319
weighted avg     0.9249    0.9091    0.9113       319

URD F1: 0.9291
ENG F1: 0.8734
Overall F1: 0.9113


## Log in to Hugging Face

This cell authenticates the Hugging Face account so the trained model can be published to a model repository.


In [ ]:
from huggingface_hub import login

login()

## Create a Hugging Face Model Repository

This cell creates a dedicated Hugging Face repository for the trained Roman Urdu–English language identification model.

In [ ]:
from huggingface_hub import create_repo

repo_name = "code-switching-codesaviours-si26-eman"

create_repo(
    repo_id=f"emanfatimaa05/{repo_name}",
    repo_type="model"
)

print("Hugging Face model repository created! 🤗✅")
print(f"https://huggingface.co/emanfatimaa05/{repo_name}")

Hugging Face model repository created! 🤗✅
https://huggingface.co/emanfatimaa05/code-switching-codesaviours-si26-eman


## Upload the Trained Model to Hugging Face

This cell uploads the trained model and tokenizer files to the Hugging Face repository for future use and sharing.

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path=MODEL_DIR,
    repo_id="emanfatimaa05/code-switching-codesaviours-si26-eman",
    repo_type="model"
)

print("MODEL UPLOADED TO HUGGING FACE! 🤗✅")

MODEL UPLOADED TO HUGGING FACE! 🤗✅


# Week 7 Final Results

**Model:** XLM-RoBERTa  
**Task:** Roman Urdu–English Token Classification  
**Labels:** URD, ENG  
**Training Sentences:** 125  
**Testing Sentences:** 32  

### Evaluation Results

| Metric | Score |
|---|---:|
| URD F1 | **92.91%** |
| ENG F1 | **87.34%** |
| Overall F1 | **91.13%** |
| Accuracy | **90.91%** |

**MIX:** Not applicable — no MIX-labelled examples were present in the dataset.

**Model:** Published on Hugging Face.
